# 5주차. 카나가 지난 대화를 불러오다

**부제:** MCP SQLite 이전 대화 검색 tool: search_previous_conversations, load_conversation_messages, extract_schedules_from_history

## 0. 목표

이번 주에는 로컬 MCP server가 SQLite 대화/일정 DB를 외부 tool 서버처럼 제공한다고 가정하고, LangChain agent가 `search_previous_conversations`, `load_conversation_messages`, `extract_schedules_from_history` tool을 호출하는 흐름을 검증한다.

성취 기준:

- 프록시 토큰을 사용하는 LangChain agent 흐름을 실행할 수 있다.
- agent가 직접 DB를 읽지 않고 MCP SQLite tool contract를 통해 이전 대화 정보를 검색하는 trace를 확인할 수 있다.

![MCP](imgs/MCP.jpg)

![mcp_arc](imgs/mcp_arc.jpg)

## 1. 준비

5주차도 프록시 서버를 통해 모델 API를 호출한다. 실제 MCP server/client 구현 문제는 별도 문제 repo에서 작성하지만, 이 노트북에서는 MCP server가 노출할 tool과 같은 이름/입출력 contract를 LangChain tool로 검증한다.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

previous_conversations = [
    {"conversation_id": "conv-a", "title": "A 일정 공유", "last_message": "A는 다음 주 화요일 15:00 가능", "members": ["A"]},
    {"conversation_id": "conv-b", "title": "B 일정 공유", "last_message": "B는 다음 주 화요일 15:00 가능", "members": ["B"]},
    {"conversation_id": "conv-c", "title": "C 일정 공유", "last_message": "C는 다음 주 수요일 10:00보다 화요일 15:00 선호", "members": ["C"]},
]
conversation_messages = {
    "conv-a": [{"role": "assistant", "content": "A는 다음 주 화요일 15:00 가능해요."}],
    "conv-b": [{"role": "assistant", "content": "B는 다음 주 화요일 15:00 가능해요."}],
    "conv-c": [{"role": "assistant", "content": "C는 다음 주 수요일 10:00보다 화요일 15:00를 선호해요."}],
}
show_json({"model": OPENAI_MODEL, "mcp_sqlite_tools": ["search_previous_conversations", "load_conversation_messages", "extract_schedules_from_history"]})

{
  "model": "openai/gpt-4.1-mini",
  "mcp_sqlite_tools": [
    "search_previous_conversations",
    "load_conversation_messages",
    "extract_schedules_from_history"
  ]
}


## 2. 개념

- MCP는 DB 접근 권한을 agent 내부 코드에서 분리한다. 
- 실제 구현에서는 MCP server가 SQLite를 읽고, agent는 MCP tool을 호출한다. 
- 이 노트북에서는 같은 tool contract를 LangChain `@tool`로 노출해 agent 선택과 trace를 검증한다.

## 3. 기본 개념 실습

이전 대화 검색 tool 세 개를 LangChain tool로 정의한다.

In [2]:
@tool("search_previous_conversations", description="SQLite 이전 대화 목록에서 멤버나 질의와 관련된 대화를 검색한다.")
def search_previous_conversations(query: str, members: list[str] | None = None) -> str:
    """Search previous conversation summaries."""
    members = members or []
    hits = [
        row for row in previous_conversations
        if not members or any(member in row["members"] for member in members) or query in row["last_message"]
    ]
    return json.dumps({"hits": hits}, ensure_ascii=False)


@tool("load_conversation_messages", description="conversation_id에 해당하는 이전 대화 메시지를 불러온다.")
def load_conversation_messages(conversation_id: str) -> str:
    """Load messages for one previous conversation."""
    return json.dumps({"conversation_id": conversation_id, "messages": conversation_messages.get(conversation_id, [])}, ensure_ascii=False)


@tool("extract_schedules_from_history", description="이전 대화 메시지에서 멤버별 가능 일정 후보를 추출한다.")
def extract_schedules_from_history(conversation_ids: list[str]) -> str:
    """Extract schedule hints from previous conversation messages."""
    schedules = []
    for conversation_id in conversation_ids:
        for message in conversation_messages.get(conversation_id, []):
            text = message["content"]
            if "15:00" in text:
                member = conversation_id.replace("conv-", "").upper()
                schedules.append({"conversation_id": conversation_id, "member": member, "date_hint": "다음 주 화요일", "start_time": "15:00", "source": text})
    return json.dumps({"schedules": schedules}, ensure_ascii=False)

show_json(json.loads(search_previous_conversations.invoke({"query": "다음 주 회의", "members": ["A", "B", "C"]})))

{
  "hits": [
    {
      "conversation_id": "conv-a",
      "title": "A 일정 공유",
      "last_message": "A는 다음 주 화요일 15:00 가능",
      "members": [
        "A"
      ]
    },
    {
      "conversation_id": "conv-b",
      "title": "B 일정 공유",
      "last_message": "B는 다음 주 화요일 15:00 가능",
      "members": [
        "B"
      ]
    },
    {
      "conversation_id": "conv-c",
      "title": "C 일정 공유",
      "last_message": "C는 다음 주 수요일 10:00보다 화요일 15:00 선호",
      "members": [
        "C"
      ]
    }
  ]
}


## 4. 카나메이트 확장 예제

Agent에게 MCP SQLite tool만 제공한다. Agent는 이전 대화에서 팀원 가능 시간을 찾기 위해 필요한 tool을 직접 선택해야 한다.

In [3]:
mcp_history_agent = create_agent(
    model=make_model(900),
    tools=[search_previous_conversations, load_conversation_messages, extract_schedules_from_history],
    system_prompt=(
        "너는 카나메이트 이전 대화 검색 agent다. 사용자가 팀원들의 과거 일정이나 가능 시간을 물으면 "
        "반드시 search_previous_conversations를 먼저 호출하고, 반환된 conversation_id 목록을 모아 "
        "extract_schedules_from_history를 두 번째로 호출한 뒤 그 결과를 근거로 답한다. "
        "load_conversation_messages는 원문 확인이 필요할 때만 선택적으로 호출한다."
    ),
)

request = "팀원 A/B/C와 다음 주 회의 시간을 잡으려면 이전 대화에서 가능한 시간을 찾아줘."
result = mcp_history_agent.invoke({"messages": [{"role": "user", "content": request}]})
trace = extract_tool_trace(result)
print(final_text(result))
show_json(trace)

팀원 A, B, C 모두 다음 주 화요일 15:00에 회의가 가능합니다. C는 수요일 10:00보다 화요일 15:00를 선호한다고 하니, 다음 주 화요일 15:00로 회의 일정을 잡는 것이 좋겠습니다.
[
  {
    "event": "tool_call",
    "tool_name": "search_previous_conversations",
    "arguments": {
      "query": "회의 시간",
      "members": [
        "A",
        "B",
        "C"
      ]
    }
  },
  {
    "event": "tool_result",
    "tool_name": "search_previous_conversations",
    "content": "{\"hits\": [{\"conversation_id\": \"conv-a\", \"title\": \"A 일정 공유\", \"last_message\": \"A는 다음 주 화요일 15:00 가능\", \"members\": [\"A\"]}, {\"conversation_id\": \"conv-b\", \"title\": \"B 일정 공유\", \"last_message\": \"B는 다음 주 화요일 15:00 가능\", \"members\": [\"B\"]}, {\"conversation_id\": \"conv-c\", \"title\": \"C 일정 공유\", \"last_message\": \"C는 다음 주 수요일 10:00보다 화요일 15:00 선호\", \"members\": [\"C\"]}]}"
  },
  {
    "event": "tool_call",
    "tool_name": "extract_schedules_from_history",
    "arguments": {
      "conversation_ids": [
        "conv-a",
        "conv-b",
        "conv-c"
 

## 5. 확장 예제 실행

trace에서 MCP SQLite tool 호출 순서와 arguments를 확인한다.

In [4]:
called_tools = [event["tool_name"] for event in trace if event["event"] == "tool_call"]
assert called_tools[:2] == ["search_previous_conversations", "extract_schedules_from_history"]
assert "extract_schedules_from_history" in called_tools
show_json({"called_tools": called_tools})

{
  "called_tools": [
    "search_previous_conversations",
    "extract_schedules_from_history"
  ]
}


## 6. 회고

확인 질문:

1. Agent가 SQLite DB를 직접 읽지 않고 MCP tool을 호출하게 하는 이유는 무엇인가?
2. 대화 목록 검색과 메시지 로드는 왜 다른 tool이어야 하는가?
3. 이전 대화에서 일정 후보를 추출할 때 반드시 남겨야 하는 필드는 무엇인가?

작은 응용 과제: 팀원 한 명의 가능 시간이 겹치지 않는 케이스를 추가하고 trace가 어떻게 달라지는지 확인한다.